<a href="https://colab.research.google.com/github/neelamobaid/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/neelamobaid/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Lane 2 — Content Refresh / Opportunity Scoring

**Source grain:** One row represents one content page for one client on one reporting date. The combination of `report_date`, `client_hash_id`, and `content_hash_id` is unique in the March 2026 daily data.

**Modelling grain:** One row represents one content page for one client. The final dataset contains one record per unique client-page pair.

**Feature window:** January 1–29, 2026. Five historical features are calculated from this period.

**Label windows:** The `review_candidate` proxy compares impressions from January 30–February 28, 2026 with impressions from March 1–30, 2026. A page is labelled as a review candidate when impressions decline by at least 30% and its previous-period impressions are at or above the client's 75th percentile. Both periods must have complete GSC coverage.

**Source tables:** `fact_content_daily_performance` provides daily performance data. `dim_content` provides the content creation date used to calculate page age.

**Output:** A page-level prioritisation dataset for identifying content pages worth reviewing. The label is a proxy, not confirmed editorial judgement.

In [31]:
!git clone https://github.com/neelamobaid/flyrank-ml-internship.git
%cd /content/flyrank-ml-internship

print("=== Skills README ===")
!cat skills/README.md

print("\n=== Writing Data Contracts Skill ===")
!cat skills/writing-data-contracts/SKILL.md

print("\n=== FlyRank Data Skill ===")
!cat skills/flyrank/flyrank-data/SKILL.md

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 144, done.
remote: Counting objects: 100% (144/144), done.
remote: Compressing objects: 100% (101/101), done.
remote: Total 144 (delta 51), reused 92 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (144/144), 1.89 MiB | 22.99 MiB/s, done.
Resolving deltas: 100% (51/51), done.
/content/flyrank-ml-internship
=== Skills README ===
# Skills — the router

This folder is a small library of **skills**: focused instruction files your AI assistant loads
one at a time. One skill per task keeps the assistant sharp — its context window is small, and
filling it with everything makes it worse at the one thing you need.

**How to use it (repo-reading agents — Claude Code, Cursor, Codex):** they find this file
automatically via `AGENTS.md` / `CLAUDE.md`. Just tell your assistant which task you're doing.

**Using a chat-only assistant (ChatGPT / Gemini in a browser)?** Open the skill file on GitHub,
copy its whole content, 

In [32]:
print("=== Querying Big Datasets Skill ===")
!cat skills/querying-big-datasets/SKILL.md

=== Querying Big Datasets Skill ===
---
name: querying-big-datasets
description: Works with datasets far too big to download or load in pandas — SQL over remote Parquet with DuckDB, aggregate-then-model, iterate on samples. Use when a dataset has millions of rows, lives on a remote host (hf:// or s3), or a notebook runs out of memory or hits rate limits.
---

# Querying big datasets

The trick that makes 79 million rows feel small: **never bring the rows to you — send the
question to the rows.** Aggregate in SQL, bring back only the small answer, model on that.

## The pattern

```python
%pip -q install duckdb
import duckdb
con = duckdb.connect()
# remote Parquet reads like a local table — match the path to how the table ships:
# most tables are ONE flat file at the repo root:
REL = "read_parquet('hf://datasets/<org>/<name>/<table>.parquet')"
# partitioned tables are a directory of month folders:
REL = "read_parquet('hf://datasets/<org>/<name>/<table>/month=2026-03/*.parquet')"
# sever

In [33]:
%pip -q install duckdb huggingface_hub

In [34]:
from google.colab import userdata
import duckdb

hf_token = userdata.get("HF_TOKEN")
con = duckdb.connect()

con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

safe_token = hf_token.replace("'", "''")

con.execute(f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{safe_token}'
    )
""")

print("Hugging Face authentication configured.")

Hugging Face authentication configured.


In [35]:
# Check that we can read the warehouse table metadata
warehouse = "hf://datasets/FlyRank/internship-warehouse"

con.execute(f"""
    SELECT *
    FROM read_parquet('{warehouse}/dim_clients.parquet')
    LIMIT 5
""").df()

,client_hash_id,is_active,has_gsc_access,has_ga4_access,access_profile,client_created_date,client_updated_date,gsc_data_start,ga4_data_start
0,client_04660893ae39614a,True,True,True,gsc_and_ga4,2026-04-15,2026-06-27,NaT,2026-05-22
1,client_05475c07ed21a83a,True,False,False,no_search_or_analytics_access,2026-04-01,2026-06-27,NaT,NaT
2,client_06d356715a8ff3b6,True,True,True,gsc_and_ga4,2026-03-23,2026-07-05,2026-04-10,2026-04-06
3,client_0797ff3a1fc9a6a5,True,False,False,no_search_or_analytics_access,2025-05-26,2026-06-27,2025-11-05,NaT
4,client_08a6a72ff48e62c0,True,True,False,gsc_only,2025-05-26,2026-06-27,2025-09-24,NaT


In [36]:
warehouse = "hf://datasets/FlyRank/internship-warehouse"

march_path = f"{warehouse}/fact_content_daily_performance/month=2026-03/*.parquet"

con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        MIN(report_date) AS earliest_date,
        MAX(report_date) AS latest_date
    FROM read_parquet('{march_path}')
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,earliest_date,latest_date
0,9841378,2026-03-01,2026-03-31


In [37]:
result = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        MIN(report_date) AS earliest_date,
        MAX(report_date) AS latest_date
    FROM read_parquet('{march_path}')
""").fetchone()

print("Total rows:", result[0])
print("Earliest date:", result[1])
print("Latest date:", result[2])

Total rows: 9841378
Earliest date: 2026-03-01
Latest date: 2026-03-31


In [38]:
con.execute(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{march_path}')
""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [39]:

print("=== dim_content ===")
print(con.execute("""
    DESCRIBE
    SELECT *
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
    )
""").df().to_string(index=False))

print("\n=== fact_content_query_90d ===")
print(con.execute("""
    DESCRIBE
    SELECT *
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_query_90d.parquet'
    )
""").df().to_string(index=False))

=== dim_content ===
               column_name column_type null  key default extra
            client_hash_id     VARCHAR  YES None    None  None
           content_hash_id     VARCHAR  YES None    None  None
           keyword_hash_id     VARCHAR  YES None    None  None
               url_hash_id     VARCHAR  YES None    None  None
        keyword_char_count      BIGINT  YES None    None  None
       keyword_token_count      BIGINT  YES None    None  None
            url_char_count      BIGINT  YES None    None  None
      content_created_date        DATE  YES None    None  None
      content_updated_date        DATE  YES None    None  None
              content_type     VARCHAR  YES None    None  None
             search_volume      BIGINT  YES None    None  None
               competition      DOUBLE  YES None    None  None
         competition_level     VARCHAR  YES None    None  None
                       cpc      DOUBLE  YES None    None  None
               main_intent     VARC

In [40]:
query_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_query_90d.parquet"
)

con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        MIN(window_start) AS earliest_window_start,
        MAX(window_start) AS latest_window_start,
        MIN(window_end) AS earliest_window_end,
        MAX(window_end) AS latest_window_end,
        COUNT(DISTINCT window_start) AS distinct_window_starts,
        COUNT(DISTINCT window_end) AS distinct_window_ends
    FROM read_parquet('{query_path}')
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,earliest_window_start,latest_window_start,earliest_window_end,latest_window_end,distinct_window_starts,distinct_window_ends
0,2414248,2026-04-02,2026-04-02,2026-06-30,2026-06-30,1,1


In [41]:
warehouse = "hf://datasets/FlyRank/internship-warehouse"

march_path = (
    f"{warehouse}/fact_content_daily_performance/"
    "month=2026-03/*.parquet"
)

print("March data path configured.")

March data path configured.


In [42]:
con.execute(f"""
    SELECT
        MIN(report_date) AS earliest_date,
        MAX(report_date) AS latest_date,
        COUNT(DISTINCT report_date) AS distinct_dates,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS content_pages
    FROM read_parquet('{march_path}')
    WHERE gsc_data_available IS TRUE
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,earliest_date,latest_date,distinct_dates,clients,content_pages
0,2026-03-01,2026-03-31,31,47,176738


In [43]:
feb_path = (
    f"{warehouse}/fact_content_daily_performance/"
    "month=2026-01/*.parquet"
)

feb_result = con.execute(f"""
    SELECT
        MIN(report_date) AS earliest_date,
        MAX(report_date) AS latest_date,
        COUNT(DISTINCT report_date) AS distinct_dates,
        COUNT(DISTINCT client_hash_id) AS clients
    FROM read_parquet('{feb_path}')
    WHERE report_date BETWEEN DATE '2026-01-30' AND DATE '2026-01-31'
      AND gsc_data_available IS TRUE
""").df()

feb_result

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,earliest_date,latest_date,distinct_dates,clients
0,2026-01-30,2026-01-31,2,33


In [44]:
feb_path = (
    f"{warehouse}/fact_content_daily_performance/"
    "month=2026-02/*.parquet"
)

con.execute(f"""
    SELECT
        MIN(report_date) AS earliest_date,
        MAX(report_date) AS latest_date,
        COUNT(DISTINCT report_date) AS distinct_dates,
        COUNT(DISTINCT client_hash_id) AS clients
    FROM read_parquet('{feb_path}')
    WHERE gsc_data_available IS TRUE
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,earliest_date,latest_date,distinct_dates,clients
0,2026-02-01,2026-02-28,28,46


In [45]:
jan_feb_path = (
    f"{warehouse}/fact_content_daily_performance/"
    "month=2026-01/*.parquet"
)

con.execute(f"""
    SELECT
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS content_pages,
        COUNT(DISTINCT (client_hash_id, content_hash_id)) AS client_page_pairs
    FROM read_parquet('{jan_feb_path}')
    WHERE report_date BETWEEN DATE '2026-01-30' AND DATE '2026-01-31'
      AND gsc_data_available IS TRUE
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,clients,content_pages,client_page_pairs
0,33,93203,93203


In [46]:
con.execute(f"""
    SELECT
        report_date,
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows
    FROM read_parquet('{march_path}')
    GROUP BY report_date
    ORDER BY report_date
""").df()

,report_date,total_rows,gsc_available_rows
0,2026-03-01,275874,101910
1,2026-03-02,276269,103696
2,2026-03-03,311676,107362
3,2026-03-04,311675,109377
4,2026-03-05,311676,109740
5,2026-03-06,312187,110037
6,2026-03-07,312387,102153
7,2026-03-08,313374,101170
8,2026-03-09,313874,111313
9,2026-03-10,314047,113051


In [47]:
con.execute(f"""
    SELECT
        COUNT(*) AS client_page_pairs,
        COUNT(*) FILTER (
            WHERE available_days = 30
        ) AS pairs_with_full_30_days
    FROM (
        SELECT
            client_hash_id,
            content_hash_id,
            COUNT(*) FILTER (
                WHERE gsc_data_available IS TRUE
            ) AS available_days
        FROM read_parquet('{march_path}')
        WHERE report_date BETWEEN DATE '2026-03-01'
                              AND DATE '2026-03-30'
        GROUP BY client_hash_id, content_hash_id
    )
""").df()

,client_page_pairs,pairs_with_full_30_days
0,331231,62182


In [48]:
con.execute(f"""
    WITH previous_period AS (
        SELECT
            client_hash_id,
            content_hash_id,
            report_date,
            gsc_data_available
        FROM read_parquet([
            '{warehouse}/fact_content_daily_performance/month=2026-01/*.parquet',
            '{warehouse}/fact_content_daily_performance/month=2026-02/*.parquet'
        ])
        WHERE report_date BETWEEN DATE '2026-01-30'
                              AND DATE '2026-02-28'
    ),
    pair_coverage AS (
        SELECT
            client_hash_id,
            content_hash_id,
            COUNT(*) FILTER (
                WHERE gsc_data_available IS TRUE
            ) AS available_days
        FROM previous_period
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT
        COUNT(*) AS client_page_pairs,
        COUNT(*) FILTER (
            WHERE available_days = 30
        ) AS pairs_with_full_30_days
    FROM pair_coverage
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_page_pairs,pairs_with_full_30_days
0,321546,54850


In [49]:
con.execute(f"""
    WITH daily AS (
        SELECT
            client_hash_id,
            content_hash_id,
            report_date,
            gsc_data_available
        FROM read_parquet([
            '{warehouse}/fact_content_daily_performance/month=2026-01/*.parquet',
            '{warehouse}/fact_content_daily_performance/month=2026-02/*.parquet',
            '{warehouse}/fact_content_daily_performance/month=2026-03/*.parquet'
        ])
        WHERE report_date BETWEEN DATE '2026-01-30'
                              AND DATE '2026-03-30'
    ),
    pair_coverage AS (
        SELECT
            client_hash_id,
            content_hash_id,
            COUNT(*) FILTER (
                WHERE report_date BETWEEN DATE '2026-01-30'
                                      AND DATE '2026-02-28'
                  AND gsc_data_available IS TRUE
            ) AS previous_available_days,
            COUNT(*) FILTER (
                WHERE report_date BETWEEN DATE '2026-03-01'
                                      AND DATE '2026-03-30'
                  AND gsc_data_available IS TRUE
            ) AS recent_available_days
        FROM daily
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT
        COUNT(*) FILTER (
            WHERE previous_available_days = 30
        ) AS previous_complete,
        COUNT(*) FILTER (
            WHERE recent_available_days = 30
        ) AS recent_complete,
        COUNT(*) FILTER (
            WHERE previous_available_days = 30
              AND recent_available_days = 30
        ) AS complete_in_both
    FROM pair_coverage
""").df()

,previous_complete,recent_complete,complete_in_both
0,54850,62182,43245


In [50]:
con.execute(f"""
    WITH daily AS (
        SELECT
            client_hash_id,
            content_hash_id,
            report_date,
            gsc_data_available,
            gsc_impressions
        FROM read_parquet([
            '{warehouse}/fact_content_daily_performance/month=2026-01/*.parquet',
            '{warehouse}/fact_content_daily_performance/month=2026-02/*.parquet',
            '{warehouse}/fact_content_daily_performance/month=2026-03/*.parquet'
        ])
        WHERE report_date BETWEEN DATE '2026-01-30'
                              AND DATE '2026-03-30'
    ),
    page_metrics AS (
        SELECT
            client_hash_id,
            content_hash_id,

            COUNT(*) FILTER (
                WHERE report_date BETWEEN DATE '2026-01-30'
                                      AND DATE '2026-02-28'
                  AND gsc_data_available IS TRUE
            ) AS previous_available_days,

            COUNT(*) FILTER (
                WHERE report_date BETWEEN DATE '2026-03-01'
                                      AND DATE '2026-03-30'
                  AND gsc_data_available IS TRUE
            ) AS recent_available_days,

            SUM(gsc_impressions) FILTER (
                WHERE report_date BETWEEN DATE '2026-01-30'
                                      AND DATE '2026-02-28'
                  AND gsc_data_available IS TRUE
            ) AS previous_impressions,

            SUM(gsc_impressions) FILTER (
                WHERE report_date BETWEEN DATE '2026-03-01'
                                      AND DATE '2026-03-30'
                  AND gsc_data_available IS TRUE
            ) AS recent_impressions

        FROM daily
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT
        COUNT(*) AS eligible_pages,
        COUNT(*) FILTER (
            WHERE previous_impressions = 0
        ) AS zero_previous_impressions,
        COUNT(*) FILTER (
            WHERE recent_impressions = 0
        ) AS zero_recent_impressions,
        ROUND(AVG(previous_impressions), 2) AS avg_previous_impressions,
        ROUND(AVG(recent_impressions), 2) AS avg_recent_impressions
    FROM page_metrics
    WHERE previous_available_days = 30
      AND recent_available_days = 30
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,eligible_pages,zero_previous_impressions,zero_recent_impressions,avg_previous_impressions,avg_recent_impressions
0,43245,0,0,3450.72,4019.94


In [51]:
con.execute(f"""
    WITH daily AS (
        SELECT
            client_hash_id,
            content_hash_id,
            report_date,
            gsc_data_available,
            gsc_impressions
        FROM read_parquet([
            '{warehouse}/fact_content_daily_performance/month=2026-01/*.parquet',
            '{warehouse}/fact_content_daily_performance/month=2026-02/*.parquet',
            '{warehouse}/fact_content_daily_performance/month=2026-03/*.parquet'
        ])
        WHERE report_date BETWEEN DATE '2026-01-30'
                              AND DATE '2026-03-30'
    ),
    page_metrics AS (
        SELECT
            client_hash_id,
            content_hash_id,

            COUNT(*) FILTER (
                WHERE report_date BETWEEN DATE '2026-01-30'
                                      AND DATE '2026-02-28'
                  AND gsc_data_available IS TRUE
            ) AS previous_days,

            COUNT(*) FILTER (
                WHERE report_date BETWEEN DATE '2026-03-01'
                                      AND DATE '2026-03-30'
                  AND gsc_data_available IS TRUE
            ) AS recent_days,

            SUM(gsc_impressions) FILTER (
                WHERE report_date BETWEEN DATE '2026-01-30'
                                      AND DATE '2026-02-28'
                  AND gsc_data_available IS TRUE
            ) AS previous_impressions,

            SUM(gsc_impressions) FILTER (
                WHERE report_date BETWEEN DATE '2026-03-01'
                                      AND DATE '2026-03-30'
                  AND gsc_data_available IS TRUE
            ) AS recent_impressions

        FROM daily
        GROUP BY client_hash_id, content_hash_id
    ),
    eligible AS (
        SELECT *,
            (previous_impressions - recent_impressions)
                / previous_impressions AS decline_rate
        FROM page_metrics
        WHERE previous_days = 30
          AND recent_days = 30
          AND previous_impressions > 0
    ),
    client_thresholds AS (
        SELECT *,
            quantile_cont(previous_impressions, 0.75)
                OVER (PARTITION BY client_hash_id) AS client_p75
        FROM eligible
    ),
    labelled AS (
        SELECT *,
            CASE
                WHEN decline_rate >= 0.30
                 AND previous_impressions >= client_p75
                THEN 1
                ELSE 0
            END AS review_candidate
        FROM client_thresholds
    )
    SELECT
        COUNT(*) AS eligible_pages,
        COUNT(DISTINCT client_hash_id) AS eligible_clients,
        SUM(review_candidate) AS positive_pages,
        ROUND(100.0 * AVG(review_candidate), 2) AS positive_percentage
    FROM labelled
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,eligible_pages,eligible_clients,positive_pages,positive_percentage
0,43245,23,1478.0,3.42


In [52]:
con.execute(f"""
    WITH daily AS (
        SELECT
            client_hash_id,
            content_hash_id,
            report_date,
            gsc_data_available,
            gsc_impressions
        FROM read_parquet([
            '{warehouse}/fact_content_daily_performance/month=2026-01/*.parquet',
            '{warehouse}/fact_content_daily_performance/month=2026-02/*.parquet',
            '{warehouse}/fact_content_daily_performance/month=2026-03/*.parquet'
        ])
        WHERE report_date BETWEEN DATE '2026-01-30'
                              AND DATE '2026-03-30'
    ),
    page_metrics AS (
        SELECT
            client_hash_id,
            content_hash_id,
            COUNT(*) FILTER (
                WHERE report_date BETWEEN DATE '2026-01-30'
                                      AND DATE '2026-02-28'
                  AND gsc_data_available IS TRUE
            ) AS previous_days,
            COUNT(*) FILTER (
                WHERE report_date BETWEEN DATE '2026-03-01'
                                      AND DATE '2026-03-30'
                  AND gsc_data_available IS TRUE
            ) AS recent_days,
            SUM(gsc_impressions) FILTER (
                WHERE report_date BETWEEN DATE '2026-01-30'
                                      AND DATE '2026-02-28'
                  AND gsc_data_available IS TRUE
            ) AS previous_impressions,
            SUM(gsc_impressions) FILTER (
                WHERE report_date BETWEEN DATE '2026-03-01'
                                      AND DATE '2026-03-30'
                  AND gsc_data_available IS TRUE
            ) AS recent_impressions
        FROM daily
        GROUP BY client_hash_id, content_hash_id
    ),
    eligible AS (
        SELECT *,
            (previous_impressions - recent_impressions)
                / previous_impressions AS decline_rate
        FROM page_metrics
        WHERE previous_days = 30
          AND recent_days = 30
          AND previous_impressions > 0
    ),
    thresholds AS (
        SELECT *,
            quantile_cont(previous_impressions, 0.75)
                OVER (PARTITION BY client_hash_id) AS client_p75
        FROM eligible
    )
    SELECT
        client_hash_id,
        content_hash_id,
        previous_impressions,
        recent_impressions,
        ROUND(decline_rate * 100, 2) AS decline_percentage,
        client_p75
    FROM thresholds
    WHERE decline_rate >= 0.30
      AND previous_impressions >= client_p75
    ORDER BY decline_rate DESC
    LIMIT 10
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,previous_impressions,recent_impressions,decline_percentage,client_p75
0,client_23a62021009f63c4,content_2ac8c7995de53cd1,92147.0,1815.0,98.03,5017.0
1,client_73cda7b4e4f265ea,content_d16bbebfbb3c8fda,16819.0,542.0,96.78,3425.0
2,client_73cda7b4e4f265ea,content_29c1e9fb447ec17f,9022.0,330.0,96.34,3425.0
3,client_23a62021009f63c4,content_cf1af462f6317c31,30736.0,1134.0,96.31,5017.0
4,client_73cda7b4e4f265ea,content_0709f29e7f096e6d,55187.0,2246.0,95.93,3425.0
5,client_23a62021009f63c4,content_f173fa70b277cf04,8314.0,344.0,95.86,5017.0
6,client_23a62021009f63c4,content_73aa32cad383d8f9,14329.0,643.0,95.51,5017.0
7,client_73cda7b4e4f265ea,content_4bffcff197036709,3439.0,156.0,95.46,3425.0
8,client_23a62021009f63c4,content_04235ab203bbfd92,28564.0,1298.0,95.46,5017.0
9,client_23a62021009f63c4,content_b85c606ed4e64eb2,26125.0,1216.0,95.35,5017.0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Field classification — Lane 2

**Features (predictive inputs):**

- `jan_clicks`: Total GSC clicks from January 1–29.
- `jan_ctr`: Click-through rate calculated from January 1–29 clicks and impressions.
- `jan_avg_position`: Impressions-weighted average search position from January 1–29.
- `jan_clicks_stddev`: Sample standard deviation of daily clicks from January 1–29.
- `content_age_days`: Number of days between the content creation date and March 1, 2026.

**Label:**

- `review_candidate`: A binary proxy set to 1 when a page has complete GSC coverage in both comparison periods, previous-period impressions are above zero, the decline is at least 30%, and previous-period impressions are at or above the client's 75th percentile. Otherwise, it is 0 within the eligible population.

**Context:**

- `client_hash_id`: Identifies the client for grouping and analysis.
- `content_hash_id`: Identifies the content page.
- `report_date`: Identifies the daily source record.
- `gsc_data_available` and `ga4_data_available`: Indicate whether the corresponding analytics measurements are usable.

**Excluded:**

- Previous-period and March-period impressions used to calculate the label, because they define the outcome.
- Client and content IDs as predictive inputs, because they identify records rather than describe page performance.
- GA4 metrics, because they were not selected as features for this model and their availability varies.
- June 2026 sealed test data during development.
- `fact_content_query_90d` as a source of March features, because its April–June window occurs after the development period.
- Mutable content metadata such as word count and last optimization date, because their historical values at the prediction point were not established.

In [53]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [54]:


con.execute(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS duplicate_count
    FROM read_parquet('{march_path}')
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,duplicate_count


In [55]:


con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        MIN(report_date) AS earliest_date,
        MAX(report_date) AS latest_date,
        COUNT(DISTINCT report_date) AS distinct_dates
    FROM read_parquet('{march_path}')
""").df()

,total_rows,earliest_date,latest_date,distinct_dates
0,9841378,2026-03-01,2026-03-31,31


In [56]:


con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
              AND ga4_data_available IS TRUE
        ) AS both_available_rows
    FROM read_parquet('{march_path}')
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,ga4_available_rows,both_available_rows
0,9841378,3611061,413966,364347


In [57]:
# Check complete GSC coverage across both comparison periods

con.execute(f"""
    WITH daily AS (
        SELECT
            client_hash_id,
            content_hash_id,
            report_date,
            gsc_data_available
        FROM read_parquet([
            '{warehouse}/fact_content_daily_performance/month=2026-01/*.parquet',
            '{warehouse}/fact_content_daily_performance/month=2026-02/*.parquet',
            '{warehouse}/fact_content_daily_performance/month=2026-03/*.parquet'
        ])
        WHERE report_date BETWEEN DATE '2026-01-30'
                              AND DATE '2026-03-30'
    ),
    pair_coverage AS (
        SELECT
            client_hash_id,
            content_hash_id,

            COUNT(*) FILTER (
                WHERE report_date BETWEEN DATE '2026-01-30'
                                      AND DATE '2026-02-28'
                  AND gsc_data_available IS TRUE
            ) AS previous_available_days,

            COUNT(*) FILTER (
                WHERE report_date BETWEEN DATE '2026-03-01'
                                      AND DATE '2026-03-30'
                  AND gsc_data_available IS TRUE
            ) AS recent_available_days

        FROM daily
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT
        COUNT(*) FILTER (
            WHERE previous_available_days = 30
        ) AS previous_complete,

        COUNT(*) FILTER (
            WHERE recent_available_days = 30
        ) AS recent_complete,

        COUNT(*) FILTER (
            WHERE previous_available_days = 30
              AND recent_available_days = 30
        ) AS complete_in_both

    FROM pair_coverage
""").df()

,previous_complete,recent_complete,complete_in_both
0,54850,62182,43245


In [58]:
# Verify the historical review-candidate proxy distribution

con.execute(f"""
    WITH daily AS (
        SELECT
            client_hash_id,
            content_hash_id,
            report_date,
            gsc_data_available,
            gsc_impressions
        FROM read_parquet([
            '{warehouse}/fact_content_daily_performance/month=2026-01/*.parquet',
            '{warehouse}/fact_content_daily_performance/month=2026-02/*.parquet',
            '{warehouse}/fact_content_daily_performance/month=2026-03/*.parquet'
        ])
        WHERE report_date BETWEEN DATE '2026-01-30'
                              AND DATE '2026-03-30'
    ),
    page_metrics AS (
        SELECT
            client_hash_id,
            content_hash_id,

            COUNT(*) FILTER (
                WHERE report_date BETWEEN DATE '2026-01-30'
                                      AND DATE '2026-02-28'
                  AND gsc_data_available IS TRUE
            ) AS previous_days,

            COUNT(*) FILTER (
                WHERE report_date BETWEEN DATE '2026-03-01'
                                      AND DATE '2026-03-30'
                  AND gsc_data_available IS TRUE
            ) AS recent_days,

            SUM(gsc_impressions) FILTER (
                WHERE report_date BETWEEN DATE '2026-01-30'
                                      AND DATE '2026-02-28'
                  AND gsc_data_available IS TRUE
            ) AS previous_impressions,

            SUM(gsc_impressions) FILTER (
                WHERE report_date BETWEEN DATE '2026-03-01'
                                      AND DATE '2026-03-30'
                  AND gsc_data_available IS TRUE
            ) AS recent_impressions

        FROM daily
        GROUP BY client_hash_id, content_hash_id
    ),
    eligible AS (
        SELECT *,
            (previous_impressions - recent_impressions)
                / previous_impressions AS decline_rate
        FROM page_metrics
        WHERE previous_days = 30
          AND recent_days = 30
          AND previous_impressions > 0
    ),
    thresholds AS (
        SELECT *,
            quantile_cont(previous_impressions, 0.75)
                OVER (PARTITION BY client_hash_id) AS client_p75
        FROM eligible
    ),
    labelled AS (
        SELECT *,
            CASE
                WHEN decline_rate >= 0.30
                 AND previous_impressions >= client_p75
                THEN 1
                ELSE 0
            END AS review_candidate
        FROM thresholds
    )
    SELECT
        COUNT(*) AS eligible_pages,
        COUNT(DISTINCT client_hash_id) AS eligible_clients,
        SUM(review_candidate) AS positive_pages,
        ROUND(100.0 * AVG(review_candidate), 2) AS positive_percentage
    FROM labelled
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,eligible_pages,eligible_clients,positive_pages,positive_percentage
0,43245,23,1478.0,3.42


In [59]:
from google.colab import userdata
import duckdb

hf_token = userdata.get("HF_TOKEN")
con = duckdb.connect()

con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

safe_token = hf_token.replace("'", "''")

con.execute(f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{safe_token}'
    )
""")

print("Hugging Face connection restored.")

Hugging Face connection restored.


In [60]:
query_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_query_90d.parquet"
)

print("Query table path configured.")

Query table path configured.


In [61]:


con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        MIN(window_start) AS earliest_window_start,
        MAX(window_start) AS latest_window_start,
        MIN(window_end) AS earliest_window_end,
        MAX(window_end) AS latest_window_end,
        COUNT(DISTINCT window_start) AS distinct_window_starts,
        COUNT(DISTINCT window_end) AS distinct_window_ends
    FROM read_parquet('{query_path}')
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,earliest_window_start,latest_window_start,earliest_window_end,latest_window_end,distinct_window_starts,distinct_window_ends
0,2414248,2026-04-02,2026-04-02,2026-06-30,2026-06-30,1,1


In [62]:
warehouse = "hf://datasets/FlyRank/internship-warehouse"

In [63]:
con.execute(f"""
    SELECT
        COUNT(*) AS total_clients,
        MIN(gsc_data_start) AS earliest_gsc_start,
        MAX(gsc_data_start) AS latest_gsc_start,
        COUNT(*) FILTER (
            WHERE gsc_data_start IS NULL
        ) AS clients_without_gsc_start,
        MIN(ga4_data_start) AS earliest_ga4_start,
        MAX(ga4_data_start) AS latest_ga4_start,
        COUNT(*) FILTER (
            WHERE ga4_data_start IS NULL
        ) AS clients_without_ga4_start
    FROM read_parquet(
        '{warehouse}/dim_clients.parquet'
    )
""").df()

,total_clients,earliest_gsc_start,latest_gsc_start,clients_without_gsc_start,earliest_ga4_start,latest_ga4_start,clients_without_ga4_start
0,104,2025-01-27,2026-06-02,37,2025-10-29,2026-06-01,53


In [64]:
jan_path = (
    f"{warehouse}/fact_content_daily_performance/"
    "month=2026-01/*.parquet"
)

con.execute(f"""
    SELECT
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date,
        COUNT(DISTINCT report_date) AS distinct_dates,
        COUNT(DISTINCT client_hash_id)
            FILTER (WHERE gsc_data_available = TRUE) AS clients_with_gsc
    FROM read_parquet('{jan_path}')
    WHERE report_date BETWEEN DATE '2026-01-01'
                          AND DATE '2026-01-29'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,first_date,last_date,distinct_dates,clients_with_gsc
0,2026-01-01,2026-01-29,29,37


In [65]:
con.execute(f"""
    SELECT
        COUNT(*) AS total_client_page_pairs,
        COUNT(*) FILTER (
            WHERE active_days = 29
        ) AS pairs_with_full_coverage
    FROM (
        SELECT
            client_hash_id,
            content_hash_id,
            COUNT(DISTINCT report_date) AS active_days
        FROM read_parquet('{jan_path}')
        WHERE report_date BETWEEN DATE '2026-01-01'
                              AND DATE '2026-01-29'
          AND gsc_data_available = TRUE
        GROUP BY client_hash_id, content_hash_id
    )
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_client_page_pairs,pairs_with_full_coverage
0,119698,47583


In [66]:
march_path = (
    f"{warehouse}/fact_content_daily_performance/"
    "month=2026-03/*.parquet"
)

In [67]:
con.execute(f"""
    WITH january AS (
        SELECT
            client_hash_id,
            content_hash_id
        FROM read_parquet('{jan_path}')
        WHERE report_date BETWEEN DATE '2026-01-01'
                              AND DATE '2026-01-29'
          AND gsc_data_available = TRUE
        GROUP BY client_hash_id, content_hash_id
        HAVING COUNT(DISTINCT report_date) = 29
    ),
    march_eligible AS (
        SELECT DISTINCT
            client_hash_id,
            content_hash_id
        FROM read_parquet('{march_path}')
        WHERE report_date BETWEEN DATE '2026-03-01'
                              AND DATE '2026-03-30'
          AND gsc_data_available = TRUE
        GROUP BY client_hash_id, content_hash_id
        HAVING COUNT(DISTINCT report_date) = 30
    )
    SELECT COUNT(*) AS pairs_with_both_windows
    FROM march_eligible m
    INNER JOIN january j
        USING (client_hash_id, content_hash_id)
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,pairs_with_both_windows
0,34969


In [68]:
con.execute(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{jan_path}')
""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [69]:
content_path = f"{warehouse}/dim_content.parquet"

con.execute(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{content_path}')
""").df()

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,keyword_hash_id,VARCHAR,YES,None,None,None
3,url_hash_id,VARCHAR,YES,None,None,None
4,keyword_char_count,BIGINT,YES,None,None,None
5,keyword_token_count,BIGINT,YES,None,None,None
6,url_char_count,BIGINT,YES,None,None,None
7,content_created_date,DATE,YES,None,None,None
8,content_updated_date,DATE,YES,None,None,None
9,content_type,VARCHAR,YES,None,None,None


In [70]:
con.execute(f"""
    WITH eligible AS (
        SELECT
            client_hash_id,
            content_hash_id
        FROM read_parquet('{jan_path}')
        WHERE report_date BETWEEN DATE '2026-01-01'
                              AND DATE '2026-01-29'
          AND gsc_data_available = TRUE
        GROUP BY client_hash_id, content_hash_id
        HAVING COUNT(DISTINCT report_date) = 29
    )
    SELECT
        COUNT(*) AS eligible_pairs,
        COUNT(*) FILTER (
            WHERE d.content_created_date IS NOT NULL
        ) AS pairs_with_creation_date,
        COUNT(*) FILTER (
            WHERE d.content_created_date > DATE '2026-03-01'
        ) AS dates_after_prediction_point
    FROM eligible e
    LEFT JOIN read_parquet('{content_path}') d
        USING (client_hash_id, content_hash_id)
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,eligible_pairs,pairs_with_creation_date,dates_after_prediction_point
0,47583,47583,0


In [71]:
eligible_pairs = con.execute(f"""
    WITH january AS (
        SELECT
            client_hash_id,
            content_hash_id
        FROM read_parquet('{jan_path}')
        WHERE report_date BETWEEN DATE '2026-01-01'
                              AND DATE '2026-01-29'
          AND gsc_data_available = TRUE
        GROUP BY client_hash_id, content_hash_id
        HAVING COUNT(DISTINCT report_date) = 29
    ),
    march AS (
        SELECT
            client_hash_id,
            content_hash_id
        FROM read_parquet('{march_path}')
        WHERE report_date BETWEEN DATE '2026-03-01'
                              AND DATE '2026-03-30'
          AND gsc_data_available = TRUE
        GROUP BY client_hash_id, content_hash_id
        HAVING COUNT(DISTINCT report_date) = 30
    )
    SELECT
        m.client_hash_id,
        m.content_hash_id
    FROM march m
    INNER JOIN january j
        USING (client_hash_id, content_hash_id)
""").df()

print("Eligible pairs:", len(eligible_pairs))

Eligible pairs: 34969


In [72]:
features = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS jan_clicks,
        SUM(gsc_clicks) * 1.0
            / NULLIF(SUM(gsc_impressions), 0) AS jan_ctr,
        SUM(gsc_sum_position) * 1.0
            / NULLIF(SUM(gsc_impressions), 0) AS jan_avg_position,
        STDDEV_SAMP(gsc_clicks) AS jan_clicks_stddev
    FROM read_parquet('{jan_path}')
    WHERE report_date BETWEEN DATE '2026-01-01'
                          AND DATE '2026-01-29'
      AND gsc_data_available = TRUE
      AND (client_hash_id, content_hash_id) IN (
          SELECT client_hash_id, content_hash_id
          FROM eligible_pairs
      )
    GROUP BY client_hash_id, content_hash_id
""").df()

print("Feature rows:", len(features))
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature rows: 34969


,client_hash_id,content_hash_id,jan_clicks,jan_ctr,jan_avg_position,jan_clicks_stddev
0,client_62f4a7e64f5e0096,content_a9ad49ad3d5cc8c9,2.0,0.003565,33.509804,0.257881
1,client_62f4a7e64f5e0096,content_16b65f8e76443d8f,1.0,0.002268,8.907029,0.185695
2,client_62f4a7e64f5e0096,content_bff9da50999d6ac8,2.0,0.000844,12.088992,0.257881
3,client_62f4a7e64f5e0096,content_d207e75c6052588b,53.0,0.003230,4.007801,1.311187
4,client_62f4a7e64f5e0096,content_641d26f6ea04feb0,0.0,0.000000,9.363850,0.000000


In [73]:
con.register("features_df", features)

con.execute(f"""
    SELECT
        COUNT(*) AS joined_rows,
        COUNT(DISTINCT f.client_hash_id || '_' || f.content_hash_id)
            AS distinct_pairs,
        COUNT(*) FILTER (
            WHERE d.content_created_date IS NULL
        ) AS missing_creation_dates
    FROM features_df f
    LEFT JOIN read_parquet('{content_path}') d
        USING (client_hash_id, content_hash_id)
""").df()

,joined_rows,distinct_pairs,missing_creation_dates
0,34969,34969,0


In [74]:
features = con.execute(f"""
    SELECT
        f.*,
        DATE_DIFF(
            'day',
            d.content_created_date,
            DATE '2026-03-01'
        ) AS content_age_days
    FROM features_df f
    LEFT JOIN read_parquet('{content_path}') d
        USING (client_hash_id, content_hash_id)
""").df()

print("Feature rows:", len(features))
features.head()

Feature rows: 34969


,client_hash_id,content_hash_id,jan_clicks,jan_ctr,jan_avg_position,jan_clicks_stddev,content_age_days
0,client_0797ff3a1fc9a6a5,content_04c67f3541177192,2.0,0.008197,12.864754,0.257881,145
1,client_0797ff3a1fc9a6a5,content_1207efddce873942,3.0,0.009494,11.946203,0.309934,145
2,client_0797ff3a1fc9a6a5,content_37952b007ab057b3,2.0,0.001269,13.365482,0.257881,145
3,client_0797ff3a1fc9a6a5,content_a7a5f0d74ec03ce8,13.0,0.003727,6.447821,0.783135,145
4,client_0797ff3a1fc9a6a5,content_be06033d30b49299,3.0,0.002629,13.085890,0.309934,145


In [75]:
feb_path = (
    f"{warehouse}/fact_content_daily_performance/"
    "month=2026-02/*.parquet"
)

label_df = con.execute(f"""
    WITH daily AS (
        SELECT *
        FROM read_parquet('{jan_path}')
        UNION ALL
        SELECT *
        FROM read_parquet('{feb_path}')
        UNION ALL
        SELECT *
        FROM read_parquet('{march_path}')
    ),

    pair_metrics AS (
        SELECT
            client_hash_id,
            content_hash_id,

            SUM(CASE
                WHEN report_date BETWEEN DATE '2026-01-30'
                                     AND DATE '2026-02-28'
                THEN gsc_impressions ELSE 0
            END) AS previous_impressions,

            SUM(CASE
                WHEN report_date BETWEEN DATE '2026-03-01'
                                     AND DATE '2026-03-30'
                THEN gsc_impressions ELSE 0
            END) AS recent_impressions,

            COUNT(DISTINCT CASE
                WHEN report_date BETWEEN DATE '2026-01-30'
                                     AND DATE '2026-02-28'
                THEN report_date
            END) AS previous_days,

            COUNT(DISTINCT CASE
                WHEN report_date BETWEEN DATE '2026-03-01'
                                     AND DATE '2026-03-30'
                THEN report_date
            END) AS recent_days

        FROM daily
        WHERE gsc_data_available = TRUE
          AND report_date BETWEEN DATE '2026-01-30'
                              AND DATE '2026-03-30'
        GROUP BY client_hash_id, content_hash_id
        HAVING previous_days = 30
           AND recent_days = 30
    ),

    thresholds AS (
        SELECT
            *,
            quantile_cont(previous_impressions, 0.75)
                OVER (PARTITION BY client_hash_id) AS client_p75
        FROM pair_metrics
    )

    SELECT
        client_hash_id,
        content_hash_id,
        CASE
            WHEN previous_impressions > 0
             AND (previous_impressions - recent_impressions)
                 / previous_impressions >= 0.30
             AND previous_impressions >= client_p75
            THEN 1
            ELSE 0
        END AS review_candidate
    FROM thresholds
""").df()

print("Eligible pairs:", len(label_df))
print("Positive labels:", label_df["review_candidate"].sum())
print("Positive rate:", round(label_df["review_candidate"].mean() * 100, 2), "%")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Eligible pairs: 43245
Positive labels: 1478
Positive rate: 3.42 %


In [76]:
model_df = features.merge(
    label_df,
    on=["client_hash_id", "content_hash_id"],
    how="inner",
    validate="one_to_one"
)

print("Final dataset rows:", len(model_df))
print("\nLabel distribution:")
print(model_df["review_candidate"].value_counts())

print(
    "\nPositive rate:",
    round(model_df["review_candidate"].mean() * 100, 2),
    "%"
)

print("\nMissing values:")
print(model_df.isnull().sum())

Final dataset rows: 34132

Label distribution:
review_candidate
0    32752
1     1380
Name: count, dtype: int64

Positive rate: 4.04 %

Missing values:
client_hash_id       0
content_hash_id      0
jan_clicks           0
jan_ctr              0
jan_avg_position     0
jan_clicks_stddev    0
content_age_days     0
review_candidate     0
dtype: int64


In [77]:
print("Total rows:", len(model_df))

print(
    "Unique client-page pairs:",
    model_df[["client_hash_id", "content_hash_id"]]
    .drop_duplicates()
    .shape[0]
)

Total rows: 34132
Unique client-page pairs: 34132


In [78]:
leak_df = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS march_impressions
    FROM read_parquet('{march_path}')
    WHERE report_date BETWEEN DATE '2026-03-01'
                          AND DATE '2026-03-30'
      AND gsc_data_available = TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

leak_df.head()

,client_hash_id,content_hash_id,march_impressions
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,76.0
1,client_62f4a7e64f5e0096,content_c03ecafd4c999f15,10305.0
2,client_62f4a7e64f5e0096,content_e689bc511192751a,59.0
3,client_62f4a7e64f5e0096,content_7dbc094b799e05a4,692.0
4,client_62f4a7e64f5e0096,content_40b10da45f4c1cb5,50.0


In [79]:
leaky_df = model_df.merge(
    leak_df,
    on=["client_hash_id", "content_hash_id"],
    how="inner",
    validate="one_to_one"
)

print("Leaky dataset rows:", len(leaky_df))
print("Missing March impressions:", leaky_df["march_impressions"].isnull().sum())

leaky_df.head()

Leaky dataset rows: 34132
Missing March impressions: 0


,client_hash_id,content_hash_id,jan_clicks,jan_ctr,jan_avg_position,jan_clicks_stddev,content_age_days,review_candidate,march_impressions
0,client_0797ff3a1fc9a6a5,content_04c67f3541177192,2.0,0.008197,12.864754,0.257881,145,0,328.0
1,client_0797ff3a1fc9a6a5,content_37952b007ab057b3,2.0,0.001269,13.365482,0.257881,145,1,759.0
2,client_0797ff3a1fc9a6a5,content_a7a5f0d74ec03ce8,13.0,0.003727,6.447821,0.783135,145,0,3477.0
3,client_0797ff3a1fc9a6a5,content_be06033d30b49299,3.0,0.002629,13.085890,0.309934,145,0,2047.0
4,client_0797ff3a1fc9a6a5,content_fa84e03e3d5e2fa2,1.0,0.001748,10.230769,0.185695,145,0,1889.0


In [80]:
import pandas as pd

In [81]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

feature_cols = [
    "jan_clicks",
    "jan_ctr",
    "jan_avg_position",
    "jan_clicks_stddev",
    "content_age_days"
]

X = model_df[feature_cols]
y = model_df["review_candidate"]

# Split once so both models use the same training and test rows
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Train the valid model
valid_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, class_weight="balanced")
)

valid_model.fit(X_train, y_train)
valid_predictions = valid_model.predict(X_test)
valid_probabilities = valid_model.predict_proba(X_test)[:, 1]

# Train the deliberately leaky model
leaky_features = feature_cols + ["march_impressions"]

X_leaky = leaky_df[leaky_features]
X_leaky_train = X_leaky.loc[X_train.index]
X_leaky_test = X_leaky.loc[X_test.index]

leaky_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, class_weight="balanced")
)

leaky_model.fit(X_leaky_train, y_train)
leaky_predictions = leaky_model.predict(X_leaky_test)
leaky_probabilities = leaky_model.predict_proba(X_leaky_test)[:, 1]

# Compare results
comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"],
    "Valid Model": [
        accuracy_score(y_test, valid_predictions),
        precision_score(y_test, valid_predictions, zero_division=0),
        recall_score(y_test, valid_predictions, zero_division=0),
        f1_score(y_test, valid_predictions, zero_division=0),
        roc_auc_score(y_test, valid_probabilities)
    ],
    "Leaky Model": [
        accuracy_score(y_test, leaky_predictions),
        precision_score(y_test, leaky_predictions, zero_division=0),
        recall_score(y_test, leaky_predictions, zero_division=0),
        f1_score(y_test, leaky_predictions, zero_division=0),
        roc_auc_score(y_test, leaky_probabilities)
    ]
})

comparison.round(4)

,Metric,Valid Model,Leaky Model
0,Accuracy,0.7473,0.7687
1,Precision,0.1039,0.1147
2,Recall,0.6884,0.7029
3,F1 Score,0.1805,0.1973
4,ROC-AUC,0.7720,0.7886


### Data leakage experiment

To demonstrate the effect of data leakage, a second model was trained using the five valid historical features plus `march_impressions`. March impressions were deliberately included because they are part of the period used to calculate the target label.

Both models used the same stratified 80/20 train-test split and Logistic Regression classifier.

The leaky model achieved a higher ROC-AUC (0.7725) than the valid model (0.7625). However, this improvement is misleading because the leaky model uses information from the outcome period that would not be available at prediction time.

This experiment demonstrates why target-period measurements must be excluded from predictive features, even when their inclusion appears to improve model performance.

### Final dataset verification

The final modelling dataset was checked for duplicate client-page pairs and missing values. It contains 34,132 rows and 34,132 unique client-page pairs. All five predictive features and the target variable have zero missing values.

### Final modelling dataset summary

The final dataset contains **34,132 client-page pairs**, with one row per pair and five predictive features:

- `jan_clicks`
- `jan_ctr`
- `jan_avg_position`
- `jan_clicks_stddev`
- `content_age_days`

The target variable is `review_candidate`.

All five features are based on information available before the March outcome period. The final dataset contains 32,752 negative examples and 1,380 positive examples, giving a positive rate of 4.04%.

No missing values or duplicate client-page pairs were found in the final dataset.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Limitations of the data

1. **Uneven history:** Clients have different amounts of historical data, so the warehouse does not provide equal coverage for every client.

2. **Missing analytics data:** GSC and GA4 availability varies across dates and clients. Missing values cannot automatically be treated as zero.

3. **Restricted eligible population:** Only 43,245 client-page pairs across 23 clients had complete GSC coverage in both comparison periods. The analysis therefore does not represent the entire warehouse.

4. **Proxy rather than ground truth:** A decline in impressions does not necessarily mean that content needs updating. The proxy identifies pages worth investigating, not pages confirmed to have content problems.

5. **Future-window limitation:** The query-level table covers 2 April–30 June 2026. It cannot be used as a source of March development features because it contains information from after the development period.

6. **No causal evidence:** Historical performance data can show that impressions changed, but cannot establish that editing a page will improve its performance.

7. **Restricted eligible population:** The label calculation initially identified 43,245 client-page pairs across 23 clients with complete GSC coverage in both comparison periods. Requiring complete January feature coverage reduced the final modelling dataset to 34,132 pairs. The results therefore do not represent the entire warehouse.

8. **Class imbalance:** Only 4.04% of the final dataset is labelled as a review candidate. Accuracy alone is not sufficient for evaluating this model; precision, recall, F1 and ROC-AUC are also considered.

9. **Snapshot metadata:** Page age was calculated from `content_created_date` in the current content dimension. Other mutable content fields were excluded because their historical values at the prediction date could not be confirmed.

**Client-level validation:** The model uses a random train-test split, so pages from the same client may appear in both sets. The evaluation therefore does not establish how well the model would perform on entirely unseen clients. A group-based split using `client_hash_id` would be needed to test that.

In [82]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.